# Notebook 03 — Benchmark Comparison: NeuMF vs NRMS vs NRAGLS

So sánh toàn diện hai mô hình trên 4 chiều:
1. **Accuracy**: AUC, MRR, nDCG@5, nDCG@10
2. **FLOPs**: Số phép toán dấu phẩy động
3. **VRAM**: Bộ nhớ GPU tiêu thụ
4. **Latency**: Độ trễ suy luận (ms)

Đo lường khi thay đổi **sequence length L** để chứng minh
O(L²) vs O(L) scaling.

In [1]:
import sys, json, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Dynamic Path Setup for Kaggle ──
sys.path.insert(0, '.')
for p in Path('/kaggle/input').rglob('utils.py'):
    sys.path.append(str(p.parent))
    break

from utils import (
    seed_everything, WORK_DIR, MODEL_DIR, SEED,
    count_parameters, measure_vram, measure_latency
)

seed_everything(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FIG_DIR = WORK_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def find_result_file(filename):
    """Tìm file kết quả trong Kaggle Input (từ các Notebook khác) hoặc Local."""
    for p in Path('/kaggle/input').rglob(filename):
        return p
    if (WORK_DIR / filename).exists():
        return WORK_DIR / filename
    return None

neumf_path = find_result_file('neumf_results.json')
nrms_path = find_result_file('nrms_results.json')
nragls_orig_path = find_result_file('nragls_original_results.json')
nragls_path = find_result_file('nragls_results.json')
nragls_adv_path = find_result_file('nragls_advanced_results.json')

neumf_res = None
if neumf_path and neumf_path.exists():
    with open(neumf_path) as f:
        neumf_res = json.load(f)
    print("NeuMF metrics:", neumf_res['metrics'])

with open(nrms_path) as f:
    nrms_res = json.load(f)

# Load Original if exists (Ablation study)
nragls_orig_res = None
if nragls_orig_path and nragls_orig_path.exists():
    with open(nragls_orig_path) as f:
        nragls_orig_res = json.load(f)
    print("NRAGLS Original metrics:", nragls_orig_res['metrics'])

with open(nragls_path) as f:
    nragls_res = json.load(f)

# Load Advanced if exists
nragls_adv_res = None
if nragls_adv_path and nragls_adv_path.exists():
    with open(nragls_adv_path) as f:
        nragls_adv_res = json.load(f)
    print("NRAGLS++ Advanced metrics:", nragls_adv_res['metrics'])

print("NRMS metrics:", nrms_res['metrics'])
print("NRAGLS+ metrics:", nragls_res['metrics'])

nragls_bert_path = find_result_file('nragls_bert_results.json')
nragls_bert_res = None
if nragls_bert_path and nragls_bert_path.exists():
    with open(nragls_bert_path) as f:
        nragls_bert_res = json.load(f)
    print("NRAGLS-BERT metrics:", nragls_bert_res['metrics'])

nragls_bert_top2_path = find_result_file('nragls_bert_top2_results.json')
nragls_bert_top2_res = None
if nragls_bert_top2_path and nragls_bert_top2_path.exists():
    with open(nragls_bert_top2_path) as f:
        nragls_bert_top2_res = json.load(f)
    print("NRAGLS-BERT (Top-2) metrics:", nragls_bert_top2_res['metrics'])

nragls_bert_all_path = find_result_file('nragls_bert_all_results.json')
nragls_bert_all_res = None
if nragls_bert_all_path and nragls_bert_all_path.exists():
    with open(nragls_bert_all_path) as f:
        nragls_bert_all_res = json.load(f)
    print("NRAGLS-BERT (All) metrics:", nragls_bert_all_res['metrics'])

nragls_bert_precompute_path = find_result_file('nragls_bert_precompute_results.json')
nragls_bert_precompute_res = None
if nragls_bert_precompute_path and nragls_bert_precompute_path.exists():
    with open(nragls_bert_precompute_path) as f:
        nragls_bert_precompute_res = json.load(f)
    print("NRAGLS-BERT (Precompute) metrics:", nragls_bert_precompute_res['metrics'])


NeuMF metrics: {'AUC': 0.549463791016167, 'MRR': 0.27691304763159175, 'nDCG@5': 0.2559329518589941, 'nDCG@10': 0.3181028369684422}
NRAGLS Original metrics: {'AUC': 0.5919613834333719, 'MRR': 0.31314425130746104, 'nDCG@5': 0.2928717294060462, 'nDCG@10': 0.3561645748459305}
NRAGLS++ Advanced metrics: {'AUC': 0.6572783706474743, 'MRR': 0.35836242140348984, 'nDCG@5': 0.3448877027082578, 'nDCG@10': 0.4071299374093923}
NRMS metrics: {'AUC': 0.6156425470834122, 'MRR': 0.3251317221491591, 'nDCG@5': 0.3096655141477109, 'nDCG@10': 0.37212618593203356}
NRAGLS+ metrics: {'AUC': 0.5951248866514484, 'MRR': 0.3152490868793249, 'nDCG@5': 0.2963987218073913, 'nDCG@10': 0.35875424717557813}
NRAGLS-BERT (Top-2) metrics: {'AUC': 0.6617805048491427, 'MRR': 0.3603770415686986, 'nDCG@5': 0.3440458734841099, 'nDCG@10': 0.40866015379464005}
NRAGLS-BERT (Precompute) metrics: {'AUC': 0.6414384299766204, 'MRR': 0.34794298006296703, 'nDCG@5': 0.3305243367626174, 'nDCG@10': 0.3937615228195848}


## 1. Accuracy Comparison

In [3]:
metrics_keys = ['AUC', 'MRR', 'nDCG@5', 'nDCG@10']

models = []
if neumf_res is not None: models.append(('NeuMF (ID-based)', neumf_res, '#808080'))
models.append(('NRMS (Baseline)', nrms_res, '#4C72B0'))
if nragls_orig_res is not None: models.append(('NRAGLS (Original)', nragls_orig_res, '#55A868'))
models.append(('NRAGLS+ (Proposed)', nragls_res, '#DD8452'))
if nragls_adv_res is not None: models.append(('NRAGLS++ (Advanced)', nragls_adv_res, '#C44E52'))
if nragls_bert_res is not None: models.append(('NRAGLS-BERT (Frozen)', nragls_bert_res, '#8172B3'))
if nragls_bert_top2_res is not None: models.append(('NRAGLS-BERT (Top-2)', nragls_bert_top2_res, '#9370DB'))
if nragls_bert_all_res is not None: models.append(('NRAGLS-BERT (All)', nragls_bert_all_res, '#8A2BE2'))
if nragls_bert_precompute_res is not None: models.append(('NRAGLS-BERT (Precomp)', nragls_bert_precompute_res, '#9932CC'))

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(metrics_keys))
w = 0.8 / len(models)
offsets = np.linspace(-w * (len(models) - 1) / 2, w * (len(models) - 1) / 2, len(models))

all_bars = []
for i, (name, res, color) in enumerate(models):
    vals = [res['metrics'][k] for k in metrics_keys]
    bars = ax.bar(x + offsets[i], vals, w, label=name, color=color, edgecolor='white')
    all_bars.append(bars)

ax.set_ylabel('Score', fontsize=13)
ax.set_title('Accuracy Comparison', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_keys, fontsize=12)
ax.legend(fontsize=10, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3)
ax.set_ylim(0, max([max([res['metrics'][k] for k in metrics_keys]) for _, res, _ in models]) * 1.25)

for bars in all_bars:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 5), textcoords='offset points', ha='center', fontsize=8, rotation=90)
plt.tight_layout()
plt.savefig(FIG_DIR / 'accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIG_DIR / 'accuracy_comparison.png'}")

Saved: /kaggle/working/dl_results/figures/accuracy_comparison.png


## 2. Training Time Comparison

In [4]:
fig, ax = plt.subplots(figsize=(9, 5))

def plot_time(res, fmt, label, color):
    if res is not None:
        y = res['epoch_times_sec']
        x = range(1, len(y) + 1)
        ax.plot(list(x), y, fmt, label=label, color=color, linewidth=2)

plot_time(neumf_res, 'v-', 'NeuMF (ID-based)', '#808080')
plot_time(nrms_res, 'o-', 'NRMS (Baseline)', '#4C72B0')
plot_time(nragls_orig_res, '^-', 'NRAGLS (Original)', '#55A868')
plot_time(nragls_res, 's-', 'NRAGLS+ (Proposed)', '#DD8452')
plot_time(nragls_adv_res, 'd-', 'NRAGLS++ (Advanced)', '#C44E52')
plot_time(nragls_bert_res, 'p-', 'NRAGLS-BERT (Frozen)', '#8172B3')
# plot_time(nragls_bert_top2_res, 'p--', 'NRAGLS-BERT (Top-2)', '#9370DB')
plot_time(nragls_bert_all_res, 'p-.', 'NRAGLS-BERT (All)', '#8A2BE2')
plot_time(nragls_bert_precompute_res, 'p:', 'NRAGLS-BERT (Precomp)', '#9932CC')

ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Time (seconds)', fontsize=13)
ax.set_title('Training Time per Epoch', fontsize=15, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
# Đảm bảo trục X chỉ hiển thị số nguyên
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(FIG_DIR / 'training_time.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIG_DIR / 'training_time.png'}")


Saved: /kaggle/working/dl_results/figures/training_time.png


## 3. Scaling Benchmark: VRAM & Latency vs Sequence Length

Re-instantiate minimal user encoders and measure resource usage
at varying sequence lengths.

In [5]:
# --- Minimal model definitions for benchmarking ---
class AdditiveAttention(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, x, mask=None):
        w = self.proj(x).squeeze(-1)
        if mask is not None: w = w.masked_fill(mask, -1e4)
        return (x * torch.softmax(w, dim=-1).unsqueeze(-1)).sum(dim=1)

class NRMSUserEncoder(nn.Module):
    """NRMS user encoder: O(L²) Self-Attention."""
    def __init__(self, dim=256, heads=16):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, batch_first=True, dropout=0.1)
        self.pool = AdditiveAttention(dim)
    def forward(self, x, mask=None):
        out, _ = self.mha(x, x, x, key_padding_mask=mask)
        return self.pool(out, mask)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        return self.scale * x / torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)

class GatedLinearAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, dim // n_heads
        self.W_q = nn.Linear(dim, dim)
        self.W_k = nn.Linear(dim, dim)
        self.W_v = nn.Linear(dim, dim)
        self.W_o = nn.Linear(dim, dim)
        self.gate = nn.Sequential(nn.Linear(dim, dim), nn.Sigmoid())
    def forward(self, x, mask=None):
        B, L, D = x.shape
        H, d = self.n_heads, self.head_dim
        phi = lambda t: F.elu(t) + 1.0
        Q = phi(self.W_q(x)).view(B, L, H, d)
        K = phi(self.W_k(x)).view(B, L, H, d)
        V = self.W_v(x).view(B, L, H, d)
        g = self.gate(x).view(B, L, H, d)
        K, V = K * g, V * g
        if mask is not None:
            m = (~mask).float().view(B, L, 1, 1)
            K, V = K * m, V * m
        KV = torch.einsum('blhd,blhe->bhde', K, V)
        out = torch.einsum('blhd,bhde->blhe', Q, KV)
        denom = torch.einsum('blhd,bhd->blh', Q, K.sum(1)).unsqueeze(-1)
        out = out / (denom + 1e-6)
        return self.W_o(out.reshape(B, L, D))

class SGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        h = dim * 4
        self.W1, self.W2, self.Wo = nn.Linear(dim, h), nn.Linear(dim, h), nn.Linear(h, dim)
    def forward(self, x):
        return self.Wo(F.silu(self.W1(x)) * self.W2(x))

class NRAGLSUserEncoder(nn.Module):
    """NRAGLS user encoder: O(L) Gated Linear Attention."""
    def __init__(self, dim=256, heads=16, n_layers=2):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'norm1': RMSNorm(dim), 'gla': GatedLinearAttention(dim, heads),
                'norm2': RMSNorm(dim), 'ffn': SGLU(dim),
            }))
        self.pool = AdditiveAttention(dim)
    def forward(self, x, mask=None):
        for L in self.layers:
            x = x + L['gla'](L['norm1'](x), mask)
            x = x + L['ffn'](L['norm2'](x))
        return self.pool(x, mask)

In [6]:
SEQ_LENGTHS = [10, 20, 30, 50, 100, 200, 300, 500, 1000]
DIM = 256
BATCH = 32

if DEVICE == 'cuda':
    nrms_vram, nragls_vram = [], []
    nrms_lat, nragls_lat = [], []

    for L in SEQ_LENGTHS:
        print(f"\n--- Benchmarking L={L} ---")
        x_input = torch.randn(BATCH, L, DIM, device=DEVICE)

        # NRMS
        enc_nrms = NRMSUserEncoder(DIM, 16).to(DEVICE)
        fn_nrms = lambda m: m(x_input)
        v = measure_vram(enc_nrms, fn_nrms, DEVICE)
        t_mean, t_std = measure_latency(enc_nrms, fn_nrms, DEVICE)
        nrms_vram.append(v)
        nrms_lat.append(t_mean)
        print(f"  NRMS:   VRAM={v:.1f}MB, Latency={t_mean:.2f}±{t_std:.2f}ms")
        del enc_nrms; torch.cuda.empty_cache()

        # NRAGLS
        enc_gla = NRAGLSUserEncoder(DIM, 16, 2).to(DEVICE)
        fn_gla = lambda m: m(x_input)
        v = measure_vram(enc_gla, fn_gla, DEVICE)
        t_mean, t_std = measure_latency(enc_gla, fn_gla, DEVICE)
        nragls_vram.append(v)
        nragls_lat.append(t_mean)
        print(f"  NRAGLS: VRAM={v:.1f}MB, Latency={t_mean:.2f}±{t_std:.2f}ms")
        del enc_gla; torch.cuda.empty_cache()

    # ── Plot VRAM scaling ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(SEQ_LENGTHS, nrms_vram, 'o-', label='NRMS O(L²)',
                 color='#4C72B0', linewidth=2, markersize=8)
    axes[0].plot(SEQ_LENGTHS, nragls_vram, 's-', label='NRAGLS O(L)',
                 color='#DD8452', linewidth=2, markersize=8)
    axes[0].set_xlabel('Sequence Length (L)', fontsize=13)
    axes[0].set_ylabel('Peak VRAM (MB)', fontsize=13)
    axes[0].set_title('GPU Memory vs Sequence Length', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(SEQ_LENGTHS, nrms_lat, 'o-', label='NRMS O(L²)',
                 color='#4C72B0', linewidth=2, markersize=8)
    axes[1].plot(SEQ_LENGTHS, nragls_lat, 's-', label='NRAGLS O(L)',
                 color='#DD8452', linewidth=2, markersize=8)
    axes[1].set_xlabel('Sequence Length (L)', fontsize=13)
    axes[1].set_ylabel('Inference Latency (ms)', fontsize=13)
    axes[1].set_title('Latency vs Sequence Length', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'scaling_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIG_DIR / 'scaling_benchmark.png'}")

    # ── Efficiency ratio plot ──
    speedup = [n/g for n, g in zip(nrms_lat, nragls_lat)]
    vram_ratio = [n/g for n, g in zip(nrms_vram, nragls_vram)]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(np.arange(len(SEQ_LENGTHS)) - 0.15, speedup, 0.3,
           label='Speed-up (×)', color='#55A868', edgecolor='white')
    ax.bar(np.arange(len(SEQ_LENGTHS)) + 0.15, vram_ratio, 0.3,
           label='VRAM reduction (×)', color='#C44E52', edgecolor='white')
    ax.set_xticks(range(len(SEQ_LENGTHS)))
    ax.set_xticklabels([str(l) for l in SEQ_LENGTHS])
    ax.set_xlabel('Sequence Length (L)', fontsize=13)
    ax.set_ylabel('Ratio (NRMS / NRAGLS)', fontsize=13)
    ax.set_title('NRAGLS Efficiency Gain over NRMS', fontsize=15, fontweight='bold')
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'efficiency_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("CUDA not available — skipping VRAM/Latency benchmarks")


--- Benchmarking L=10 ---
  NRMS:   VRAM=12.4MB, Latency=0.44±0.02ms
  NRAGLS: VRAM=22.5MB, Latency=2.37±0.04ms

--- Benchmarking L=20 ---
  NRMS:   VRAM=14.8MB, Latency=0.59±0.01ms
  NRAGLS: VRAM=27.2MB, Latency=2.40±0.16ms

--- Benchmarking L=30 ---
  NRMS:   VRAM=17.6MB, Latency=0.45±0.02ms
  NRAGLS: VRAM=32.5MB, Latency=2.59±0.18ms

--- Benchmarking L=50 ---
  NRMS:   VRAM=25.5MB, Latency=0.61±0.01ms
  NRAGLS: VRAM=42.2MB, Latency=3.20±0.07ms

--- Benchmarking L=100 ---
  NRMS:   VRAM=49.9MB, Latency=1.40±0.02ms
  NRAGLS: VRAM=65.6MB, Latency=5.97±0.05ms

--- Benchmarking L=200 ---
  NRMS:   VRAM=126.8MB, Latency=3.75±0.02ms
  NRAGLS: VRAM=115.5MB, Latency=11.49±0.10ms

--- Benchmarking L=300 ---
  NRMS:   VRAM=244.1MB, Latency=7.50±0.03ms
  NRAGLS: VRAM=159.9MB, Latency=17.32±0.17ms

--- Benchmarking L=500 ---
  NRMS:   VRAM=608.3MB, Latency=17.98±0.07ms
  NRAGLS: VRAM=253.3MB, Latency=28.24±0.23ms

--- Benchmarking L=1000 ---
  NRMS:   VRAM=2243.6MB, Latency=62.81±0.16ms
  NRAGL

## 4. Summary Table

In [7]:
models_list = []
if neumf_res is not None: models_list.append(('NeuMF (Classic CF)', neumf_res))
models_list.append(('NRMS (Baseline)', nrms_res))
if nragls_orig_res is not None: models_list.append(('NRAGLS (Original)', nragls_orig_res))
models_list.append(('NRAGLS+ (Proposed)', nragls_res))
if nragls_adv_res is not None: models_list.append(('NRAGLS++ (Advanced)', nragls_adv_res))
if nragls_bert_res is not None: models_list.append(('NRAGLS-BERT (Frozen)', nragls_bert_res))
if nragls_bert_top2_res is not None: models_list.append(('NRAGLS-BERT (Top-2)', nragls_bert_top2_res))
if nragls_bert_all_res is not None: models_list.append(('NRAGLS-BERT (All)', nragls_bert_all_res))
if nragls_bert_precompute_res is not None: models_list.append(('NRAGLS-BERT (Precomp)', nragls_bert_precompute_res))

summary = {
    'Model': [m[0] for m in models_list],
    'Complexity': [m[1].get('complexity', 'N/A') for m in models_list],
    'Parameters': [f"{m[1].get('parameters', 0):,}" for m in models_list],
    'AUC': [f"{m[1]['metrics']['AUC']:.4f}" for m in models_list],
    'MRR': [f"{m[1]['metrics']['MRR']:.4f}" for m in models_list],
    'nDCG@5': [f"{m[1]['metrics']['nDCG@5']:.4f}" for m in models_list],
    'nDCG@10': [f"{m[1]['metrics']['nDCG@10']:.4f}" for m in models_list],
    'Avg Epoch Time': [f"{m[1].get('avg_epoch_time_sec', 0):.1f}s" for m in models_list],
}

df_summary = pd.DataFrame(summary)
print("\n" + "=" * 80)
print("FINAL COMPARISON TABLE")
print("=" * 80)
print(df_summary.to_string(index=False))
print("=" * 80)
df_summary.to_csv(WORK_DIR / 'final_comparison.csv', index=False)


FINAL COMPARISON TABLE
                Model                        Complexity Parameters    AUC    MRR nDCG@5 nDCG@10 Avg Epoch Time
   NeuMF (Classic CF) O(1) inference, but O(U+I) memory 20,400,449 0.5495 0.2769 0.2559  0.3181          12.7s
      NRMS (Baseline)                        O(L^2 * d)  2,761,538 0.6156 0.3251 0.3097  0.3721         114.9s
    NRAGLS (Original)                        O(L * d^2)  3,256,386 0.5920 0.3131 0.2929  0.3562         119.2s
   NRAGLS+ (Proposed)                        O(L * d^2)  3,256,388 0.5951 0.3152 0.2964  0.3588         124.1s
  NRAGLS++ (Advanced)                        O(L * d^2)  4,685,324 0.6573 0.3584 0.3449  0.4071          80.3s
  NRAGLS-BERT (Top-2)                        O(L * d^2) 14,843,651 0.6618 0.3604 0.3440  0.4087        7986.9s
NRAGLS-BERT (Precomp)                        O(L * d^2)    667,907 0.6414 0.3479 0.3305  0.3938          35.3s
